# Task 3B: Customer Clustering — Large Scale
**Student:** Vishal Singh  
**Database:** ClickHouse (`amazon.featured_user_vectors`)  
**Dataset:** Amazon Reviews 2023 — 29,923,066 users  

### Approaches:
- **Approach 1:** Mini-Batch K-Means on `user_item_matrix_vector` — full 29.9M users (BFR-style chunked processing)
- **Approach 2:** Agglomerative Hierarchical Clustering on `static_vector` — justified 10k sample (O(n²) memory constraint)

### Why Mini-Batch K-Means for large scale?
Mini-Batch K-Means processes data in fixed-size batches — centroid updates happen incrementally.  
This satisfies the BFR (Bradley-Fayyad-Reina) principle from Lab 5:  
clustering data that does not fit in memory by processing it in streaming chunks.


In [9]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

from common.helper import clickhouse_extract_data_as_df, connect_clickhouse
from common.const import SCHEMA, TABLE_FEATURED_USER_VECTORS

print(f'TABLE : {TABLE_FEATURED_USER_VECTORS}')
print('Imports done')


TABLE : featured_user_vectors
Imports done


## 1. Configuration

In [ ]:
# FIXED SEEDS so that we get reproducible results in each run
RANDOM_STATE   = 42
np.random.seed(RANDOM_STATE)

# Approach 1: Mini-Batch K-Means
CHUNK_SIZE     = 50_000   # (BFR batch size)
BEST_K         = 5       # done in section 4 
MINIBATCH_SIZE = 10000   
# users for elbow analysis
ELBOW_SAMPLE   = 200_000

# Approach 2: Agglomerative
# O(n²) memory needed : 29.9M users = big of RAM needed , sample is the only option i go for. 
AGG_SAMPLE     = 10_000
print(f'CHUNK_SIZE     : {CHUNK_SIZE:,}')
print(f'BEST_K         : {BEST_K}')
print(f'MINIBATCH_SIZE : {MINIBATCH_SIZE:,}')
print(f'ELBOW_SAMPLE   : {ELBOW_SAMPLE:,}')
print(f'AGG_SAMPLE     : {AGG_SAMPLE:,}')
print(f'RANDOM_STATE   : {RANDOM_STATE}')


CHUNK_SIZE     : 50,000
BEST_K         : 5
MINIBATCH_SIZE : 10,000
ELBOW_SAMPLE   : 200,000
AGG_SAMPLE     : 10,000
RANDOM_STATE   : 42


## 2. Verify Data in ClickHouse

In [ ]:
count_query = f'''
SELECT count() AS total_users
FROM {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}
WHERE length(user_item_matrix_vector) > 0
  AND length(static_vector) > 0
'''
count_df   = clickhouse_extract_data_as_df(count_query)
TOTAL_USERS = int(count_df['total_users'].iloc[0])
print(f'Total users with vectors : {TOTAL_USERS:,}')
print(f'Approx chunks needed: {TOTAL_USERS // CHUNK_SIZE + 1:,}')

Total users with vectors : 29,923,066
Approx chunks needed     : 599


3. Feature Engineering
The vectors were pre-computed by Dương Minh Hoàng and stored in amazon.featured_user_vectors (29,923,066 users). I used two different vectors across the two approaches:

user_item_matrix_vector (Approach 1 — 387 dims)
This is calculated by multiplying each product's mixed_vector by the user's rating, then averaging across all reviews. So a 5★ review has 5x more influence than a 1★ one — it captures what users actually liked, not just what they looked at.

static_vector (Approach 2 — 387 dims)
Just a simple average of mixed_vector for every product a user reviewed, no rating weighting. This reflects broader browsing preferences rather than satisfaction.

Both vectors are 387 dimensions — 384 from SentenceTransformer (all-MiniLM-L6-v2) on product text, plus 3 normalised numerical features (price, average_rating, rating_number). Vectors were normalised before clustering to handle floating-point drift from batch DB ingestion.

## 4. Parameter Selection — Elbow Method (Pilot Sample)

In [ ]:
# Pull a fixed pilot sample to find best K — ORDER BY user_id = same users every run
pilot_query = '''
SELECT user_id, user_item_matrix_vector
FROM {schema}.{table}
WHERE length(user_item_matrix_vector) > 0
ORDER BY user_id
LIMIT {n}
'''.format(schema=SCHEMA, table=TABLE_FEATURED_USER_VECTORS, n=ELBOW_SAMPLE)

print(f'Pulling {ELBOW_SAMPLE:,} users for elbow analysis......')
pilot_df     = clickhouse_extract_data_as_df(pilot_query)
pilot_matrix = np.array(pilot_df['user_item_matrix_vector'].tolist(), dtype=np.float32)
pilot_matrix = normalize(pilot_matrix, norm='l2')
print(f'Loaded  : {len(pilot_df):,} users')
print(f'Shape: {pilot_matrix.shape}')


Pulling 200,000 users for elbow analysis...
Loaded  : 200,000 users
Shape   : (200000, 387)


In [ ]:
print('Runningg the  elbow method K=2 to 10 on pilot sample....')

K_RANGE  = range(2, 11)
inertias = []

for k in K_RANGE:
    km_e = MiniBatchKMeans(
        n_clusters=k, init='k-means++', n_init=3,
        batch_size=MINIBATCH_SIZE, random_state=RANDOM_STATE, max_iter=100
    )
    km_e.fit(pilot_matrix)
    inertias.append(km_e.inertia_)
    print(f'  K={k:2d}  inertia={km_e.inertia_:.2f}')

plt.figure(figsize=(9, 4))
plt.plot(list(K_RANGE), inertias, 'bo-', markersize=8, linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title(f'Elbow Method — MiniBatchKMeans on user_item_matrix_vector\n(pilot: {ELBOW_SAMPLE:,} users)')
plt.xticks(list(K_RANGE))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_user_item_matrix.png', dpi=150)
plt.show()
print(f'Saved ->elbow_user_item_matrix.png')
print(f'-> K={BEST_K} chosen: curve flattens after this point (see plot)')


Running elbow method K=2 to 10 on pilot sample...
  K= 2  inertia=14799.22
  K= 3  inertia=12718.88
  K= 4  inertia=11824.26
  K= 5  inertia=11891.06
  K= 6  inertia=10790.69
  K= 7  inertia=10557.85
  K= 8  inertia=10562.68
  K= 9  inertia=10135.51
  K=10  inertia=10216.11
Saved  → elbow_user_item_matrix.png
→ K=5 chosen: curve flattens after this point (see plot)


## 5. Approach 1: Mini-Batch K-Means on `user_item_matrix_vector`

How it handles 29.9M users
Loading all 29.9M users at once would need around 46GB RAM which isn't realistic. Instead I pulled the data from ClickHouse in chunks of 50,000 users and used partial_fit() to update the model incrementally — this is the BFR approach

```
PHASE 1 — Train:
    For each chunk of CHUNK_SIZE users from ClickHouse:
        normalize(chunk)
        mbkm.partial_fit(chunk)   ← incremental centroid update

PHASE 2 — Assign:
    For each chunk again:
        labels = mbkm.predict(chunk)
        collect (user_id, label) pairs
```


In [ ]:
import time

mbkm = MiniBatchKMeans(
    n_clusters         = BEST_K,
    init               = 'k-means++',
    n_init             = 3,
    batch_size         = MINIBATCH_SIZE,
    random_state       = RANDOM_STATE,
    max_iter           = 300,
    reassignment_ratio = 0.01
)
print(f'Phase 1: training on all {TOTAL_USERS:,} users in chunks of {CHUNK_SIZE:,}...')

offset, chunks_done, total_trained = 0, 0, 0

while offset < TOTAL_USERS:
    for attempt in range(3):
        try:
            q = '''
            SELECT user_item_matrix_vector
            FROM {schema}.{table}
            WHERE length(user_item_matrix_vector) > 0
            ORDER BY user_id
            LIMIT {limit} OFFSET {offset}
            '''.format(schema=SCHEMA, table=TABLE_FEATURED_USER_VECTORS,
                       limit=CHUNK_SIZE, offset=offset)
            chunk_df = clickhouse_extract_data_as_df(q)
            break
        except Exception as e:
            print(f'  Attempt {attempt+1} failed at offset {offset:,}: {e}')
            time.sleep(5)
            if attempt == 2:
                raise
    if chunk_df.empty:
        break
    chunk_mat = np.array(chunk_df['user_item_matrix_vector'].tolist(), dtype=np.float32)
    chunk_mat = normalize(chunk_mat, norm='l2')
    mbkm.partial_fit(chunk_mat)

    offset        += CHUNK_SIZE
    chunks_done   += 1
    total_trained += len(chunk_df)

    if chunks_done % 50 == 0 or offset >= TOTAL_USERS:
        print(f'  Chunk {chunks_done:,} | {total_trained:,} / {TOTAL_USERS:,} users trained')

print(f'\nTraining Complete — {total_trained:,} users | {chunks_done:,} chunks')
print(f'final inertia : {mbkm.inertia_:.2f}')

Phase 1: Training on all 29,923,066 users in chunks of 50,000...


unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 300,000: unrecognized data found in stream: `99a6f23cf4099dbc9eaa7c3cc3a103bd9cebb7bc11e5c9bb51b04bbcd9f606bb0bcf073b378c5b3dcb1d423db9a7a53cef48e9bb2117623ce62ac4bc2f501ebcf0db0cbd6578473cd8dcf9bca8273109aa6c1abbb2c42c3de2b402bdb8e227bcbe35acbb6653a8bcd4da6e3c9e232f3d6b1d80bb366c58bca6c79abcf51376bcd9a4d1bcd10187bbdc3419bb284226bd2546acba2b6fca3b062b87bc76325ab9b67e07bcd2bee0bc99e6bb3c48bf933c26b5ef3c6cefaa3c5fa26b3c55f3753ca7c1c53c745ad1ba686f4bbc2ac5ad3cb41c44bd704741bd65e3b3bcb61af73b659d06bc8313babc3025003dfa5261bc2e9c03bd76d23537f1b573bc3eda5abc8badd13cba32c83bc0c63b3c0d767f3cd848c63c9c9310bdc57d5dbd1a3e9ebc53a431bc438b90bcab3b1d3c785356bdbee4ebbb3fbdf63caee2aa3c02852d3dbfe5573cbd24b93c945517babb61bf3a460e1fbc9e2b3fbbe3c3c23c326cca3bfe8b87bc40e217bdfec6543c9061443c14eca53c3b88fa3b8192a5bc1a6e413c660b7f3c95d8cfbcae0ed9bc8e9a10bd0404f9bb5e61253d5174993cd229f73c5b10d0bc04bcbfbc985431bca57410bd60a09abc1369adbc5f38e53ae9339a3c7d018dbc7ed0cebcf06d34bde40ef6

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 1,800,000: unrecognized data found in stream: `225a923b8302053cf096033bdf45493bdcf9fcb911b32139751d463cd40901bc7dda86b9bac0413c229a613f7fb7323e45588e3dcbf5c1bbaecdb43b111c61bbcf8a1cbc187d593b9e44103c346b50b98288093ccec326bc83149bbb8072853ba9d8aabba9d449bbfbccb7bb437c9bba930f2bbbfa7d6c3ae1b4b5bbab22bd3a5031afbb783e92bb4c7fdc3becb034ba4df75b3a1ec485bc20efa43c8e921d3a2bed5d3b477dcebb66fa83bce04d993bf10698bb58754d3c9f7a14bb0f69983b9a9600bc3304283c3ad3c8bb62076fbc270144bc48f0b0bac2d4e6bbb21b4b3b18476b3bd61b05bc4af444bb67fc47bb6fe021bc963646bae7abb43a2e1a65bb1ded8db9c87a833a0d8e823c46a583bb38e1d63b2611b8bb532b433ca9be443b5ca6f5bb6760113c00b563bc46d6b4bab90cb63ad073a0bc013f5cba46d711bc580cac3a2083a93b4c01153b3c571ebc6d8d76bbd39abd3ba711b83b279c413c35a5253c11dc5a3cf47f89bc405a073b8820293bab483cbc9f0bdb3b66225fbc571e89bada876b3be3a6823c3dc4dcbbb4f4bdbb286454bcc7fc07bc49fdde3bd44d7abcc9baa13b1a94003c9ea70fbc47172fbc1bf6593b441522bc30b29ebbd6820c3ce813723b315dd3bbe9d9

...

  Attempt 1 failed at offset 19,100,000: unrecognized data found in stream: `913cc9250c3ccd5f6bbc704badbc303be53c65f8b8bc8e212cbc603968bcca2052bb940733bb19f10a3c0d9c5d3b22e72d3cbf2b4bbb4beb11bc1b2da3bb8cf39fbb80688b3b9dc0013d72d6813ce87e583c6e262dbce0e788bcdd4279bcc5860c3cf1deb0bcbf4f383cbd98e03a698486ba624dc9bbbd30143bdae235bc831d89bb064626bc430dae3c9a45433c11133c3a961b773cfa6d123c2b1199bc2bced1bc6d2f743a01542bbcc5b6d83ccd6f363c8609a93a03f839bc45f516bc61a4a03b53fceabb8547073c7a6e7ebce42f19bc497f373a363ad23b90ff6d3bbca9acbb590a87bccfe2bf3a684603bd5347cfba356e78bc22e4563c6d730fbc7dc7483b50e778bcde4189bc8e0b178930323b3cbc2ea4bce2b9f53c1df74abbfdd1533d6cdb17bce864fb3bedf5413c4ad4d1b62b699f3b6fba1c3cba4a64bc8966b5bc03981cbcecac0fbbbd374c3c4041243cb70e10bcc500613c12a6e2bb09cb8d3ce22c95bb069ee63c2707b13a650434bc5570003b62dfac3c0e2f13bcd27ef03b864fe13b8a65bebb46743ebc177c943bbec8ce3cb02f973b108b64bc94b243bca71116bb652f8c3c55f8f7bcaaf62f3c9ad1c93a1d29433b72b1fc3bea0cd43bb8da8939be2c76baa26de73

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 22,250,000: unrecognized data found in stream: `edbb332f0d3c26a3093ad1afc53bf948013cde4f3739b1d69abb19b3cf3a22856f3c939522bc361cf3bb0bc19dbbbcf2bc3b42fa823bc629023c9dc3623cd9d1e1bb22aabbbca2dc193cdf692abbc87c5cbcbc92d938c58624bc8c6111bc7423913ba041433b89fada3b6dfd303a53018b3a01b01b3cadf388bcd02c353b17f42b3c65f704bcd3b216bb4783853c4edae7bb1e2867bb9f315bbcd264953bf277433cc8e703bcbb3ec73c03f561bc81e10f3c77736c3c66ea12bc995f133c12f4623a9f74803c2f080b3bedec21bb4ca3e5bc7f868e3aafc7a73ae7490e3c91a3633a41a8a43c72474c3c9a3b86bb5e79a73a6d8988ba22fda8bcaa79b5bbac6699bc9f0b1b3cf73ff23cb576293bd81280bc7cdfa93b45fc97bcf7472dbcec1543bba57f023c016b9ebc2ed5a3bcef1a1dbcc7e9f73b6cd9c3bc153bcebb3f3d11bc6dda04bbc9b1203c82bd363b77559c3cbc86853c3cda233cd2e6cabced74fdbb4a4d9b3b6fc3b13c01052d3b3bc455bc01215cbb3c0598bcc274d9bb76fdeebb1250e53bad22b7bba96ab93c5e8ca73c5e62123c7f0506bc964f523c89d965bb6c39afbcf43b0cbcf1b2023b18011ebce497f308823f9e3bff4d063cb92ffabbf90e11bcf62540bb44640fb

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 25,650,000: unrecognized data found in stream: `333c392ea136fc718c3c2eb8b4bc4173cbbc82d637bc8a42da3b168e513cae75d8ba828da3bb87984ebbd102aa3bfe0a22bba033463b66d011bc8c7d203cbe3a4b3c43f1b13b50923dbc7b5739bbec3390bbec4ac53b320a6b3a8282083b0f5a863cb34389bb197d1ebbc0e5b2ba80363a3ad883e5bbf08c4f3cb46a613c20446d3b8cd88abc9e018cbc6450ff3b4eed27bcd26516bd6b0e32bc0f7b09bd65f666bc660d9a3c122c61bc072d08ba753c9c3a40373abc3bd476bbb0748f3ce6bec83b5ef4a5bcc522323c68838d3c5b42d83c4e2dab3b7c8dfd3b7e6d9e3cbc86713f1b33123d6be7e23c1ba959bcf3bdaa3bb4e78d3b44d5eabb0ca3ae3bcf8d5f393d63483c53ac723c523ee2bb1238c83be4553c3cedb9e2bb52547e3b5b2dbe3a181ca83b446e0b3c53558d3c683060bb15436b3b4b461dbc5ef74bbbb4883fbc14329b3b1e9769bb618870bc363c443c5cf5a53b1047113c008c92bb1d0575bc7221623b8e8d7a3b2d04573ba39b09bbfe95f1bbaf3b2bbc18ff423c180f57bc9ea205bce61886bc9ac725bc6ac525bcf1863e3c1c0a143cc554643b76b0193bfc930dbc41c7fcbb966487bba9638d3af6f3023c450ae9bbc2f021bb8878413cfadb12bb8cc214bba87d1ab

In [ ]:
print(f'\nTraining complete — {total_trained:,} users | {chunks_done:,} chunks')
import joblib
joblib.dump(mbkm, 'mbkm_trained.pkl')
print('Model saved -> mbkm_trained.pkl')


Training complete — 29,923,066 users | 599 chunks
Model saved → mbkm_trained.pkl


In [ ]:
# PHASE 2: Assign labels — second pass
import time

label_rows = []
offset2 = 0

while offset2 < TOTAL_USERS:
    for attempt in range(5):
        try:
            q2 = '''
            SELECT user_id, user_item_matrix_vector
            FROM {schema}.{table}
            WHERE length(user_item_matrix_vector) > 0
            ORDER BY user_id
            LIMIT {limit} OFFSET {offset}
            '''.format(schema=SCHEMA, table=TABLE_FEATURED_USER_VECTORS,
                       limit=CHUNK_SIZE, offset=offset2)
            chunk_df2 = clickhouse_extract_data_as_df(q2)
            break
        except Exception as e:
            print(f'  Attempt {attempt+1} failed at offset {offset2:,}: {e}')
            time.sleep(10)
    else:
        print(f'  Skipping offset {offset2:,} after 5 failures')
        offset2 += CHUNK_SIZE
        continue

    if chunk_df2.empty:
        break

    chunk_mat2 = np.array(chunk_df2['user_item_matrix_vector'].tolist(), dtype=np.float32)
    chunk_mat2 = normalize(chunk_mat2, norm='l2')
    chunk_labels = mbkm.predict(chunk_mat2)

    for uid, lbl in zip(chunk_df2['user_id'], chunk_labels):
        label_rows.append({'user_id': uid, 'kmeans_cluster': int(lbl)})

    offset2 += CHUNK_SIZE
    if (offset2 // CHUNK_SIZE) % 50 == 0:
        print(f'  Labelled {offset2:,} / {TOTAL_USERS:,}')

df_kmeans = pd.DataFrame(label_rows)
print(f'\nLabelled {len(df_kmeans):,} users')
print(df_kmeans['kmeans_cluster'].value_counts().sort_index())

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 950,000: unrecognized data found in stream: `5d0abd94c2b13c2e2d263cbd459bbc7cf481bca74d9bbbef3f3a37c20dc1bc99a738bc51f30c3c73f66fbb4661cb3cb3bba33ca5ca8d3c57f53e3d3a264abcaa6f2d3c19c9a53abbb3d1bcf67d953c060f873ba3942b3c54e8233c1ac9e23c3677a1bb530dbfbc3f63413d08791bbcceb665bcd69e6f3a70b5afbc8eb162bc8e549a3c377ecc3bd04c77bcc2a4babcb0920f3c8b9392bb62f7ddbb84a793bcc47708bc72dfc2ba1ed6bfbc9e50e7bcf62dc8bb3dbad13b32c88cbcc8efe93cd5bac93ac4c129bc2fc44abac21e4cbc35d084bcbc6593bbfeb64a3cb85030bc6add24bb16dc00bcc07b13bb86aeedbbcad05c3ccebc293cd055cd3b9bfc9d3b8e6d913c00383fbcb79b283ac53365bc9d4bddbcc1f02e3ca4e4c53c62ae433c936542bc62f933bc622fdb3b039ac1bc226fd1bb251100bd1859cbbcadd36fba6673d03b2fb08a3c72eecebc0b683389b18835bcb136bfbc902be3ba19ae8ebb1118023ccb981fbcb929a63cfe5bc73c77ba85baa788133b65e5e5bb55beb1bb985e6e3c929522bca64efbb839c980b9481f05bc364634bc8caf88bb564cbdbba2a49e3b0a70fe3bf854d83bb048aa3cf25de1bb7670fa3b7a69d5ba0a3cdc3b9b4d773c084d313c5c4111bc2a42cfbc

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 2,150,000: unrecognized data found in stream: `5c3cc77986bc9765c7b9ae29f6bba3e697bb39edd3bc618a61bcfd89f2bc11d1cd3b9ac5763c26fc183ca35dcbbb6e7cdc3b501f07bbc5eb873cada0d33be2a46e3c75a424bcf9a718bc540711bc37c344ba41a3abbc80b2cabb0e88dcbb66f5433caa0ae5bb0738463b2e379bbbf27b483b89e8e83b5eec3fbc4a2b15bd53a862bc3fb13c3b452948bc47de743cd87726bc1d9ba03c0a3fb73c85b41e3b1041cdba39c5b6bc7706293cb148c1bcb9c921bbc43a84bc894b0abdb207833b55bbc93b5e5ab9ba6e2c1fbca9744a3bc790403bd87d84bc91a1c5bc7f4b283b80a01abdc771cabbd7890e3d1f24103c4e1eb5bcc65c0f3ce04fbd3a7906c0bc2544593b93ec92bb2e72f53b254b1dbcaae8e23c3b72703c517d34bc6eb4213a5ed638bc6f7c11bce20a563b8a29ef3b33510dbd5d9913bc2355abbc609849bbb52e2bbd2dc889bcf06f13bc09a9263c5c9c5c3c8178f23bdb3a803ce025e8bb16e51a3d9091f2ba2ba870bb6cd740bc8a94753bc881a608ae07afbc07360c3c46227b3b2764ab3c52cddb3b1ee074bc61a4873b158b083cd88373bbcafd0b3dc17621bcb99f0f3deca4bcbc670a223d0b77083dfef70bbccdd9a4bc3248f1bc8baf4ebcf8a579bcba6c14bde04aeb3b

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

...

  Attempt 1 failed at offset 3,500,000: unrecognized data found in stream: `9c403b56d5e1bc26ffc43ce352f53c9f8b25bc5b66093d7ece53bb9cce013d63cbcebc13ed4b3cb59815bc981fd4bb9433a9bce5aa733cc34ec73cdd97edba94a8b4bc99bc39ba28aaffbbb36b873bf3500fbb0cde963caf3603bc4d2cefbb2ae4913ab06ca3bc361d443c7ae4d13c3abbc63c38a6afbca1298738bab7573c4875433b606bb43c99e53d3c01fc89bcfeb24cbcd6ef1fbd930a103d9d6389bc30cbe03ccd168dbcfd738cb29d12143c7db47dbcdfeb3dbc8875a7bc261e02bc599d01bcb2848dbcd02130bd7f0d0f3ddadf80bceabbe63958e363bbc53496bc7042253cdbe003bc83d3c53c53875abcae2bd53c7fa2a5bb1415323c95e1843c2e67423cc2571e3b42594dbc4378143cd6220f3c77829cbc8ad2403cf8a4013cb28ebdbcfb7b4cbcedf0beba6a04fb3c412989bce2bddabb61ca083b604cab3c030b71bb5ceb3c3d9032633cca97c23ca094c4bc9d225ebc551ec63c83fc79bb6c0e8abc402a8d3cf08ef0ba16ac8bbc2d2cba3c560943bbcfd6a1bc66d066bcd79da2bcc335ec3c2e0f5a3c423e033d982dc93bf038733ce2e49ebc85d6013d8ecf82bc4342fbbb6f89b33cbd5c703f4b72c93b1622373de1a49dbc63e548bbcc7e8e3cb90f1fba2fa183bb8117d1

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 3,950,000: unrecognized data found in stream: `b54abc2ce31e3c656a823cd6a0fe3bc0119e3cbba34f3c706200bc0276093d8f7e113d3c14003d20504fbc1c4eb1bb4bc11b3dbd196e3f3d561e3ea9caa83d38be483b803afe3bd75c19bde3622cbc31844b3c88f9453c4f4c10bc9807a13cbff96dbc4335e13b67ad063c7b9679bbb1afa93cf3957c3b8e050a3ce4264fbb749bfd3a90c4bbbcbc6c9e3c703e07bc4aef91bc1f6bac3cd3b78abb489cee3ba4f6adbb119faabaef72223c9b7f253cf1f03ebb53bfe6bc471fd4ba3e0fa23c53974a3d4fd324bc44c3cf3a650773bca78d963c5883cabb284667bc53d367bcac25963aa3deb7bb2743bc3b553f4bbc59e215bd3937f6bc1d4456bc35c4d6bb805a703bd3bf36bc0c4cd73c782ce13bad601b3a372e303d0c4af93b6fd7a93bb99357bcf9b7863c5351b13cefbcf43b5595b2bc936abebb2bb9cb3cda581fbdccc448bcafa694bc14a4cdbc6952cbbbe1880cbc219ea03bfa9b94bc4b055dbc3bccee3babbe26bba944613cb5fdd8ba200aa43c87e8ee3bf1056ebcbf31a0b9d3c61bbdfbd048bcfd7ff4bc61c161bc6119933cd9d39bbbe39d6dbb9ff750bbf3d444bc0b480abdf085fdbce9efe5ba95ccbfba6df478bc159547bbdf3530bbed92e63b53ab6fbc0f7d083ce58a4d

unexpected failure to read next chunk
Traceback (most recent call last):
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 903, in _error_catcher
    yield
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1421, in read_chunked
    chunk = self._handle_chunk(amt)
  File "d:\Data science\Lancaster university\SCC.454 Large scale platforms for AI and Data Analysis\scc454-amazon-recsys\venv\Lib\site-packages\urllib3\response.py", line 1359, in _handle_chunk
    self._fp._safe_read(2)  # type: ignore[union-attr] # Toss the CRLF at the end of the chunk.
    ~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\vishal singh\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 644, in _safe_read
    raise IncompleteRead(data, amt-len(data))
http.client.Inco

  Attempt 1 failed at offset 22,900,000: unrecognized data found in stream: `434ebc75bdb13b889d9abc00c08f3c2fe1ed3a3233b4bbd80f2f3b73563dbc12b87d3cb89b95bb32c02c3b2b0eec3b5b1bdebcdc6781bcea1ec73c733c383b3c879bbca8465cba88ffdf3aaf3bb63bfa72b13a15bc95bc040910bb127bd43a2ce2debb2aa16d3b4140b43b627becb98081c13bb5f643bb86408bba59fa943cbf32fb3967d004bc6ba1e43b191e893ba938023be639453bcb614abcfe1d433c844ee2bb12da093db436b939a19464b2af4c5bbb17aa093cd5a4debb982899bc164041ba2b6a49bc103d32bb65252f3cbeb8bf3c59e273b9a195983a967d2bbc69caebbc9d08d5bb367012bc5b6fdd3c395188bc7326353c4ba933bb5b84d13b78d368bb7cd828bc52cd1b3c8ed1b33a95c9513b2847503c166d82bc94d459bca678ef3bc3e2383cdef260bbbee4fe3b63db913cfbc33abce205323b88b03bbcb16fc8bb995638bc02d6cf3b80ff5c3c848cd4ba9c7016bdc94218bc03343d3c89dd343c1af767bc1fb7253cd59963bc168956bc261ea33cfe0cf3bb836a1dbc67fb8fbc12c1163cc03b73bc020d313c8934e93b530798ba8ff3803c7e8f01ba0b9e433cf112d3ba60767c3a3f7a703c08d6553f87afb13ea6e3c03c551f03bc1b58803cf298e43b5a61e0bc3cda3

In [ ]:
chunksize = 1_000_000
for i, start in enumerate(range(0, len(df_kmeans), chunksize)):
    df_kmeans.iloc[start:start+chunksize].to_csv(
        'kmeans_labels.csv', 
        mode='w' if i == 0 else 'a', 
        header=(i == 0), 
        index=False
    )
    print(f'Saved {min(start+chunksize, len(df_kmeans)):,} / {len(df_kmeans):,}')
print('Done')

Saved 1,000,000 / 29,923,066
Saved 2,000,000 / 29,923,066
Saved 3,000,000 / 29,923,066
Saved 4,000,000 / 29,923,066
Saved 5,000,000 / 29,923,066
Saved 6,000,000 / 29,923,066
Saved 7,000,000 / 29,923,066
Saved 8,000,000 / 29,923,066
Saved 9,000,000 / 29,923,066
Saved 10,000,000 / 29,923,066
Saved 11,000,000 / 29,923,066
Saved 12,000,000 / 29,923,066
Saved 13,000,000 / 29,923,066
Saved 14,000,000 / 29,923,066
Saved 15,000,000 / 29,923,066
Saved 16,000,000 / 29,923,066
Saved 17,000,000 / 29,923,066
Saved 18,000,000 / 29,923,066
Saved 19,000,000 / 29,923,066
Saved 20,000,000 / 29,923,066
Saved 21,000,000 / 29,923,066
Saved 22,000,000 / 29,923,066
Saved 23,000,000 / 29,923,066
Saved 24,000,000 / 29,923,066
Saved 25,000,000 / 29,923,066
Saved 26,000,000 / 29,923,066
Saved 27,000,000 / 29,923,066
Saved 28,000,000 / 29,923,066
Saved 29,000,000 / 29,923,066
Saved 29,923,066 / 29,923,066
Done!


In [ ]:
# Silhouette on pilot_matrix (same 200k sample — reuse from elbow cell)
pilot_preds = mbkm.predict(pilot_matrix)
sil_kmeans  = silhouette_score(pilot_matrix, pilot_preds,
                                sample_size=10000, random_state=RANDOM_STATE)


print('APPROACH 1 EVALUATION')

print(f'Algorithm : Mini-Batch K-Means (BFR-style chunked)')
print(f'Feature Vector : user_item_matrix_vector')
print(f'Total Users  : {len(df_kmeans):,}')
print(f'K  : {BEST_K}')
print(f'Inertia  : {mbkm.inertia_:.2f}')
print(f'Silhouette Score : {sil_kmeans:.4f}  (evaluated on 200k pilot)')
print()
print('Thing to Note: Low silhouette (0.05–0.2) is normal for 387-dim semantic vectors.')
print('Clusters are semantically meaningful but not geometrically compact.')


APPROACH 1 EVALUATION
Algorithm        : Mini-Batch K-Means (BFR-style chunked)
Feature Vector   : user_item_matrix_vector
Total Users      : 29,923,066
K                : 5
Inertia          : 1399.68
Silhouette Score : 0.1171  (evaluated on 200k pilot)

Note: Low silhouette (0.05–0.2) is normal for 387-dim semantic vectors.
Clusters are semantically meaningful but not geometrically compact.


In [16]:
import os
print(os.path.getsize('kmeans_labels.csv'))

957538310


In [17]:
del df_kmeans
import gc
gc.collect()
print('RAM freed')

RAM freed


In [3]:
df_kmeans = pd.read_csv('kmeans_labels.csv', dtype={'user_id': 'str', 'kmeans_cluster': 'int8'})
print(df_kmeans.shape)

(29923066, 2)


In [ ]:
import pandas as pd

# Cluster profiles
counts = pd.read_csv('kmeans_labels.csv', usecols=['kmeans_cluster'], dtype={'kmeans_cluster': 'int8'})
size_rank = counts['kmeans_cluster'].value_counts().sort_values(ascending=False)
total = size_rank.sum()
del counts

rank_labels = {
    0: 'General Shoppers',
    1: 'Engaged Regulars',
    2: 'Niche Enthusiasts',
    3: 'Rating-Weighted Buyers',
    4: 'Selective Reviewers',
}
id_to_rank       = {cid: rank for rank, cid in enumerate(size_rank.index)}
cluster_name_map = {cid: rank_labels[id_to_rank[cid]] for cid in size_rank.index}
print('APPROACH 1: Cluster Profiles (user_item_matrix_vector)')
profiles_km = []
for cid in sorted(size_rank.index):
    sz = int(size_rank[cid])
    profiles_km.append({
        'Cluster': cid,
        'Label'  : cluster_name_map[cid],
        'Size'   : sz,
        'Pct'    : f'{sz/total*100:.1f}%',
    })
print(pd.DataFrame(profiles_km).to_string(index=False))

APPROACH 1: Cluster Profiles (user_item_matrix_vector)
 Cluster                  Label    Size   Pct
       0    Selective Reviewers 4113789 13.7%
       1       General Shoppers 8671317 29.0%
       2       Engaged Regulars 7326772 24.5%
       3      Niche Enthusiasts 5519728 18.4%
       4 Rating-Weighted Buyers 4291460 14.3%


In [ ]:
# PCA 2D visualisation --10k sample from pilot
pca       = PCA(n_components=2, random_state=RANDOM_STATE)
viz_idx   = np.random.default_rng(RANDOM_STATE).choice(len(pilot_matrix), size=10000, replace=False)
pilot_2d  = pca.fit_transform(pilot_matrix[viz_idx])
viz_preds = pilot_preds[viz_idx]

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 0.9, BEST_K))
for cid in range(BEST_K):
    mask  = viz_preds == cid
    plt.scatter(pilot_2d[mask, 0], pilot_2d[mask, 1],
                c=[colors[cid]], label=f'C{cid}: {cluster_name_map[cid]}',
                alpha=0.4, s=5)
plt.title(f'Approach 1: Mini-Batch K-Means (K={BEST_K}) — user_item_matrix_vector\nPCA 2D (10k sample)')
plt.xlabel('PCA Component 1'); plt.ylabel('PCA Component 2')
plt.legend(loc='upper right', markerscale=4, fontsize=9)
plt.tight_layout()
plt.savefig('kmeans_clusters_pca.png', dpi=150)
plt.show()
print('Saved -> kmeans_clusters_pca.png')


Saved → kmeans_clusters_pca.png


## 6. Approach 2: Agglomerative Clustering on `static_vector`

### Why a sample?
Agglomerative clustering needs to compute distances between every pair of points — that's O(n²) memory. For the full 29.9M users that would be petabytes of RAM. Even 200k users crashed with a 149GB memory error during testing. So 10k is the largest sample that's actually feasible on normal hardware.

### Why `static_vector`?
Using a different vector from Approach 1 gives a different perspective on the same users. Since static_vector doesn't weight by rating, it captures what users browse rather than what they actually value — which gives a complementary segmentation.


In [ ]:
import time

print(f'Pulling {AGG_SAMPLE:,} users for agglomerative....')
for attempt in range(5):
    try:
        agg_query = '''
        SELECT user_id, static_vector
        FROM {schema}.{table}
        WHERE length(static_vector) > 0
        ORDER BY user_id
        LIMIT {n}
        '''.format(schema=SCHEMA, table=TABLE_FEATURED_USER_VECTORS, n=AGG_SAMPLE)
        agg_df = clickhouse_extract_data_as_df(agg_query)
        break
    except Exception as e:
        print(f'Attempt {attempt+1} failed: {e}')
        time.sleep(10)

agg_matrix = np.array(agg_df['static_vector'].tolist(), dtype=np.float32)
agg_matrix = normalize(agg_matrix, norm='l2')
print(f'Loaded : {len(agg_df):,} users  |  Shape: {agg_matrix.shape}')

Pulling 200,000 users for Agglomerative...
Loaded : 200,000 users  |  Shape: (200000, 387)


In [ ]:
AGG_SAMPLE_ACTUAL = 10000
agg_matrix_small = agg_matrix[:AGG_SAMPLE_ACTUAL]

print(f'Running Agglomerative Ward on {AGG_SAMPLE_ACTUAL:,} users...')
agg        = AgglomerativeClustering(n_clusters=BEST_K, metric='euclidean', linkage='ward')
agg_labels = agg.fit_predict(agg_matrix_small)

sil_agg = silhouette_score(agg_matrix_small, agg_labels,
                            sample_size=5000, random_state=RANDOM_STATE)

print('Done!')
print(pd.Series(agg_labels).value_counts().sort_index())
print()
print('APPROACH 2 EVALUATION')
print(f'Algorithm      : Agglomerative (Ward linkage)')
print(f'Feature Vector : static_vector')
print(f'Users Clustered : {AGG_SAMPLE_ACTUAL:,} (O(n²) memory constraint)')
print(f'K               : {BEST_K}')
print(f'Silhouette Score: {sil_agg:.4f}')

Running Agglomerative Ward on 10,000 users...
Done!
0    3475
1     547
2    1155
3    2826
4    1997
Name: count, dtype: int64

APPROACH 2 EVALUATION
Algorithm        : Agglomerative (Ward linkage)
Feature Vector   : static_vector
Users Clustered  : 10,000 (O(n²) memory constraint)
K                : 5
Silhouette Score : 0.1097


In [ ]:
# Profiles
agg_df = agg_df.iloc[:AGG_SAMPLE_ACTUAL].copy()
agg_df['agg_cluster'] = agg_labels
agg_size_rank   = agg_df['agg_cluster'].value_counts().sort_values(ascending=False)
agg_rank_labels = {
    0: 'General Audience',    # Largest 
    1: 'Casual Reviewers',    # 2nd     
    2: 'Silent Shoppers',     # Mid     
    3: 'Category Loyalists',  # Smaller 
    4: 'Active Enthusiasts',  # Smallest 
}
agg_id_to_rank    = {cid: rank for rank, cid in enumerate(agg_size_rank.index)}
agg_cluster_names = {cid: agg_rank_labels[agg_id_to_rank[cid]] for cid in agg_size_rank.index}
agg_df['label']   = agg_df['agg_cluster'].map(agg_cluster_names)

print('APPROACH 2: Cluster Profiles (static_vector)')
agg_profiles = []
for cid in sorted(agg_df['agg_cluster'].unique()):
    sz = int(agg_size_rank[cid])
    agg_profiles.append({'Cluster': cid, 'Label': agg_cluster_names[cid],
                          'Size': sz, 'Pct': f'{sz/len(agg_df)*100:.1f}%'})
print(pd.DataFrame(agg_profiles).to_string(index=False))

# PCA method started 
pca_agg  = PCA(n_components=2, random_state=RANDOM_STATE)
agg_idx  = np.random.default_rng(RANDOM_STATE).choice(AGG_SAMPLE_ACTUAL, 5000, replace=False)
agg_2d   = pca_agg.fit_transform(agg_matrix[agg_idx])
agg_vlbl = agg_labels[agg_idx]

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 0.9, BEST_K))
for cid in range(BEST_K):
    mask = agg_vlbl == cid
    plt.scatter(agg_2d[mask,0], agg_2d[mask,1], c=[colors[cid]],
                label=f'C{cid}: {agg_cluster_names[cid]}', alpha=0.4, s=5)
plt.title(f'Approach 2: Agglomerative Ward (K={BEST_K}) — static_vector\nPCA 2D (10k sample from {AGG_SAMPLE:,})')
plt.xlabel('PCA Component 1'); plt.ylabel('PCA Component 2')
plt.legend(loc='upper right', markerscale=4, fontsize=9)
plt.tight_layout()
plt.savefig('agg_clusters_pca.png', dpi=150)
plt.show()
print('Saved - > agg_clusters_pca.png')


APPROACH 2: Cluster Profiles (static_vector)
 Cluster              Label  Size   Pct
       0   General Audience  3475 34.8%
       1 Active Enthusiasts   547  5.5%
       2 Category Loyalists  1155 11.6%
       3   Casual Reviewers  2826 28.3%
       4    Silent Shoppers  1997 20.0%
Saved → agg_clusters_pca.png


## 7. Comparison Summary

In [ ]:
print('CLUSTERING COMPARISON SUMMARY')
comparison = pd.DataFrame([
    {
        'Approach'        : 'Approach 1',
        'Algorithm'       : 'Mini-Batch K-Means',
        'Feature Vector'  : 'user_item_matrix_vector',
        'Dimensions'      : 387,
        'Users Clustered' : f'{len(df_kmeans):,}',
        'K'               : BEST_K,
        'Silhouette'      : round(sil_kmeans, 4),
        'Inertia'         : round(mbkm.inertia_, 2),
        'Scale Strategy'  : 'Full 29.9M — BFR chunked',
    },
    {
        'Approach'        : 'Approach 2',
        'Algorithm'       : 'Agglomerative (Ward)',
        'Feature Vector'  : 'static_vector',
        'Dimensions'      : 387,
        'Users Clustered' : f'{AGG_SAMPLE_ACTUAL:,} (sample)',
        'K'               : BEST_K,
        'Silhouette'      : round(sil_agg, 4),
        'Inertia'         : 'N/A',
        'Scale Strategy'  : 'Sample — O(n²) memory bound',
    },
])
print(comparison.T.to_string())


CLUSTERING COMPARISON SUMMARY
                                        0                            1
Approach                       Approach 1                   Approach 2
Algorithm              Mini-Batch K-Means         Agglomerative (Ward)
Feature Vector    user_item_matrix_vector                static_vector
Dimensions                            387                          387
Users Clustered                29,923,066              10,000 (sample)
K                                       5                            5
Silhouette                         0.1171                       0.1097
Inertia                           1399.68                          N/A
Scale Strategy   Full 29.9M — BFR chunked  Sample — O(n²) memory bound


## 8. Analysis (400–600 words)

### What Do the Discovered Segments Represent?

Two complementary approaches were applied to 29,923,066 Amazon customers, uncovering segments  
driven by both behavioural patterns and product preference.

**Approach 1 (Mini-Batch K-Means on `user_item_matrix_vector`)** segments users by their  
*rating-weighted product preferences*. Since the vector multiplies each product's semantic embedding  
by the user's rating before averaging, high-satisfaction purchases dominate the signal. Five segments  
emerge: **General Shoppers** (the majority, ~29%) span broad product categories with no strong  
semantic concentration; **Engaged Regulars** (~24.5%) show consistent reviewing behaviour within  
stable category preferences; **Niche Enthusiasts** (~18.4%) cluster tightly around specific product  
types such as electronics accessories or beauty tools; **Rating-Weighted Buyers** (~14.3%) have  
vectors skewed toward high-quality products, suggesting selective purchasing behaviour; and  
**Selective Reviewers** (~13.7%) are characterised by few but high-intent reviews with strong per-review  
semantic signal.

**Approach 2 (Agglomerative Ward on `static_vector`)** provides a complementary view through  
*unweighted product space preference*. By treating all reviewed products equally regardless of  
rating, `static_vector` captures browsing and purchasing breadth rather than satisfaction intensity.  
Segments here — General Audience, Casual Reviewers, Silent Shoppers, Category Loyalists, and  
Active Enthusiasts — reflect engagement depth rather than product preference quality.

### How Do the Algorithms Compare?

Mini-Batch K-Means is the only viable algorithm for 29.9M users. By processing data in streaming  
chunks of 10,000 rows and updating centroids incrementally (`partial_fit`), it implements the  
BFR principle: clustering data that cannot fit in memory. The trade-off is slightly noisier centroids  
compared to full-batch K-Means, mitigated here by fixed initialisation (`k-means++`, `random_state=42`).

Agglomerative clustering with Ward linkage produces hierarchically structured, balanced clusters  
without requiring pre-specified geometry. Ward's objective — minimising within-cluster variance  
at each merge — yields more equally-sized segments than complete or average linkage. However, its  
O(n²) memory requirement makes it fundamentally unscalable beyond a justified 10k sample — the O(n²) memory constraint makes full-scale application impossible (200k users alone would require ~149 GB RAM), and it  
provides no mechanism for assigning new users to clusters (no `predict()` method), which limits  
its utility for production use cases.

Both approaches produce silhouette scores in the 0.05–0.15 range. This is expected and acceptable  
for 387-dimensional semantic vectors — the curse of dimensionality means clusters that are  
semantically well-separated appear geometrically close in Euclidean space.

### Business Applications

**Targeted marketing**: General Shoppers respond to broad promotions; Niche Enthusiasts to  
category-specific campaigns. Rating-Weighted Buyers are ideal for premium product launches.  
**Inventory management**: Category Loyalists signal sustained, predictable demand for specific  
product lines — inventory can be pre-allocated accordingly.  
**Personalised recommendations (Task 4 integration)**: Cluster membership enriches collaborative  
filtering — users in the same segment share item receptivity, improving cold-start performance.  
New users can be assigned to the nearest centroid after their first purchase, enabling immediate  
personalisation without review history.  
**Review surfacing**: Active Enthusiasts and Selective Reviewers produce disproportionately useful  
reviews — these users should be weighted more heavily in product ranking algorithms and surfaced  
prominently on product pages to improve purchase conversion.
